# state_objective_fold_native_teacher_router_v1

Goal: close the remaining state-objective BC blocker from the previous card: the teacher path was produced by a globally selected diagnostic router. This notebook re-screens the same router grid separately inside each train fold, builds fold-native teacher paths, regenerates rebound clusters/targets, and rechecks BC label health.

External basis:
- Qlib keeps data, model, strategy, and backtest layers separate, which maps to this notebook's reusable `src` router functions plus runtime-only experiment orchestration. <https://github.com/microsoft/qlib>
- FinRL emphasizes explicit data/environment/agent/evaluation stages for DRL trading experiments, so this run separates teacher generation from later BC/PPO training. <https://github.com/AI4Finance-Foundation/FinRL>
- Backtest-overfitting/PBO literature motivates not selecting the teacher on the same global OOS path used for promotion review. <https://doi.org/10.1090/conm/729/14680>

This card can permit the next diagnostic training run, but it cannot promote a runtime model by itself.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.config import load_profile
from src.soxl_quant.app.state_conditioned_router import (
    StateConditionedRouterConfig,
    build_state_conditioned_router_frame,
    screen_state_conditioned_router_configs,
    state_conditioned_router_metrics,
)
from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveReboundClusterConfig,
    build_fold_local_state_objective_training_targets,
    discover_rebound_clusters_from_daily_path,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
)
from src.soxl_quant.policy.policy_builder import build_state_objective_warm_start_dataset_from_frame

EXPERIMENT = "state_objective_fold_native_teacher_router_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
PREVIOUS_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_native_cluster_targets_v1"
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
CARD.mkdir(parents=True, exist_ok=True)

BASE_PATH = ROOT / "runtime/execution/card/risk_budget_residual_drawdown_guard_v1/risk_budget_residual_drawdown_guard_v1_selected_soxl_daily.csv"
UPSIDE_PATHS = {
    "recovery_reentry_conditional_vol_target_v1": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/recovery_reentry_conditional_vol_target_v1_soxl_daily.csv",
    "recovery_reentry_anchor_confirm_v1": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/recovery_reentry_anchor_confirm_v1_soxl_daily.csv",
    "recovery_reentry_regime_drawdown_guard_v1": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1_soxl_daily.csv",
    "recovery_reentry_cppi_floor_v1": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1_soxl_daily.csv",
    "recovery_reentry_2021_trendbreak_cap_v1": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/recovery_reentry_2021_trendbreak_cap_v1_soxl_daily.csv",
}
missing = [str(BASE_PATH.relative_to(ROOT))] if not BASE_PATH.exists() else []
missing += [str(path.relative_to(ROOT)) for path in UPSIDE_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing router source paths:\n" + "\n".join(missing))

base_daily = pd.read_csv(BASE_PATH, parse_dates=["datetime"])
upside_daily_by_family = {family: pd.read_csv(path, parse_dates=["datetime"]) for family, path in UPSIDE_PATHS.items()}
feature_frame = pd.read_pickle(FRAME_PATH)
print({"base_rows": len(base_daily), "families": len(upside_daily_by_family), "feature_rows": len(feature_frame)})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'base_rows': 1569, 'families': 5, 'feature_rows': 2973}


## Governance setup

Closest historical success: the global `state_conditioned_rebound_risk_router_v1` found a useful diagnostic router but selected it on global evidence. Closest historical failure: the follow-up fold-local target cards could not train because target provenance still depended on that global teacher. This run is not a threshold rename: it changes the selection provenance from global OOS to fold-local train windows while keeping the same router family/grid.

In [2]:
folds = [
    {
        "name": "wf_2022_train",
        "train_start": "2020-01-02",
        "train_end": "2021-12-31",
        "eval_start": "2022-01-03",
        "eval_end": "2022-12-30",
    },
    {
        "name": "wf_2023_2024_train",
        "train_start": "2020-01-02",
        "train_end": "2022-12-30",
        "eval_start": "2023-01-03",
        "eval_end": "2024-12-31",
    },
]
GRID = {
    "upside_cap": [0.20, 0.25, 0.35, 0.45],
    "rebound_mom21": [0.30, 0.40, 0.50, 0.70],
    "rebound_volq": [0.40, 0.60, 0.75],
    "risk_cap": [0.00, 0.05, 0.10, 0.15],
    "risk_dd63": [-0.08, -0.10, -0.15, -0.20],
    "risk_mom21": [-0.05, 0.00],
    "use_semis_stress": [False, True],
    "require_semis_widen": [False, True],
}
FIXED = {
    "rebound_mom5": 0.02,
    "rebound_min_dd63": -0.35,
    "semis_ratio_z": -0.75,
    "semis_ratio_widen": -0.05,
    "semis_soxx_mom21": 0.0,
}

def iter_configs():
    keys = list(GRID)
    for values in product(*(GRID[key] for key in keys)):
        params = dict(zip(keys, values))
        params.update(FIXED)
        yield StateConditionedRouterConfig(**params)

configs = list(iter_configs())
{"folds": folds, "configs_per_family": len(configs), "total_fold_family_candidates": len(configs) * len(UPSIDE_PATHS) * len(folds)}

{'folds': [{'name': 'wf_2022_train',
   'train_start': '2020-01-02',
   'train_end': '2021-12-31',
   'eval_start': '2022-01-03',
   'eval_end': '2022-12-30'},
  {'name': 'wf_2023_2024_train',
   'train_start': '2020-01-02',
   'train_end': '2022-12-30',
   'eval_start': '2023-01-03',
   'eval_end': '2024-12-31'}],
 'configs_per_family': 6144,
 'total_fold_family_candidates': 61440}

## Fold-native router screening

For each fold and upside family, the score only sees train dates. The later eval dates are computed after selection for review and are not part of `robust_score`. Selection also requires minimum train-window rebound coverage before sorting by target gate and robust score, because the downstream state-objective target needs high-vol rebound examples rather than a teacher that optimizes by never entering rebound.

In [3]:
MIN_TRAIN_REBOUND_DAYS = 3
screen_frames = []
for fold in folds:
    for family, upside_daily in upside_daily_by_family.items():
        scored = screen_state_conditioned_router_configs(
            base_daily,
            upside_daily,
            configs,
            fold=fold["name"],
            train_start=fold["train_start"],
            train_end=fold["train_end"],
        )
        scored["family"] = family
        screen_frames.append(scored)

fold_router_screen = pd.concat(screen_frames, ignore_index=True)
fold_router_screen["rebound_coverage_gate"] = fold_router_screen["train_rebound_days"] >= MIN_TRAIN_REBOUND_DAYS
fold_router_screen = fold_router_screen.sort_values(
    ["fold", "rebound_coverage_gate", "target_gate", "robust_score", "family", "candidate_id"],
    ascending=[True, False, False, True, True, True],
).reset_index(drop=True)
fold_router_top = fold_router_screen.groupby("fold", group_keys=False).head(200).reset_index(drop=True)
fold_router_selection = fold_router_screen.groupby("fold", group_keys=False).head(1).reset_index(drop=True)
fold_router_top.to_csv(CARD / "fold_router_top_configs.csv", index=False)
fold_router_selection.to_csv(CARD / "fold_router_selection.csv", index=False)
fold_router_selection[["fold", "family", "candidate_id", "rebound_coverage_gate", "target_gate", "robust_score", "train_relative_return", "train_max_dd", "train_avg_pos", "train_rebound_days", "train_risk_days", "train_changed_days"]]

,fold,family,candidate_id,rebound_coverage_gate,target_gate,robust_score,train_relative_return,train_max_dd,train_avg_pos,train_rebound_days,train_risk_days,train_changed_days
0,wf_2022_train,recovery_reentry_conditional_vol_target_v1,5500,True,False,50130.719651,-193.898212,14.576033,0.208183,8,98,22
1,wf_2023_2024_train,recovery_reentry_conditional_vol_target_v1,380,True,True,0.331045,92.477348,14.576033,0.160822,8,267,30


## Generate selected fold-native teacher paths

The selected config is applied to the full overlapping base/upside path, then split into train and eval review windows. Train paths feed BC targets. Eval paths are only a leakage check / sanity review.

In [4]:
def _config_from_row(row: pd.Series) -> StateConditionedRouterConfig:
    fields = StateConditionedRouterConfig.__dataclass_fields__.keys()
    return StateConditionedRouterConfig(**{field: row[field] for field in fields})

def _date_filter(frame: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    dates = pd.to_datetime(frame["datetime"]).dt.normalize()
    return frame.loc[(dates >= pd.Timestamp(start).normalize()) & (dates <= pd.Timestamp(end).normalize())].reset_index(drop=True)

train_daily_paths = {}
eval_daily_paths = {}
eval_metric_rows = []
train_metric_rows = []
for row in fold_router_selection.to_dict("records"):
    fold = next(item for item in folds if item["name"] == row["fold"])
    config = _config_from_row(pd.Series(row))
    routed = build_state_conditioned_router_frame(
        base_daily,
        upside_daily_by_family[row["family"]],
        config,
    )
    train_daily = _date_filter(routed, fold["train_start"], fold["train_end"])
    eval_daily = _date_filter(routed, fold["eval_start"], fold["eval_end"])
    train_daily_paths[row["fold"]] = train_daily
    eval_daily_paths[row["fold"]] = eval_daily
    train_daily.to_csv(CARD / f"{row['fold']}_teacher_train_daily.csv", index=False)
    eval_daily.to_csv(CARD / f"{row['fold']}_teacher_eval_review_daily.csv", index=False)
    train_metrics = state_conditioned_router_metrics(train_daily)
    eval_metrics = state_conditioned_router_metrics(eval_daily)
    train_metric_rows.append({"fold": row["fold"], "family": row["family"], **train_metrics})
    eval_metric_rows.append({"fold": row["fold"], "family": row["family"], **eval_metrics})

train_metrics = pd.DataFrame(train_metric_rows)
eval_review_metrics = pd.DataFrame(eval_metric_rows)
train_metrics.to_csv(CARD / "fold_train_teacher_metrics.csv", index=False)
eval_review_metrics.to_csv(CARD / "fold_eval_review_metrics.csv", index=False)
eval_review_metrics

,fold,family,rows,return,buy_and_hold_return,relative_return,max_dd,avg_pos,turnover,rebound_days,risk_days,changed_days
0,wf_2022_train,recovery_reentry_conditional_vol_target_v1,251,-8.928575,-86.588072,77.659496,10.740376,0.069633,1.871401,8,169,13
1,wf_2023_2024_train,recovery_reentry_conditional_vol_target_v1,502,32.317004,191.773509,-159.456505,10.518580,0.149070,10.692987,0,179,45


## Rebuild targets from fold-native teacher paths

In [5]:
cluster_config = StateObjectiveReboundClusterConfig(min_rows=3)
cluster_frames = []
for fold in folds:
    name = fold["name"]
    cluster_frames.append(
        discover_rebound_clusters_from_daily_path(
            train_daily_paths[name],
            candidate=f"{EXPERIMENT}:teacher",
            fold=name,
            train_start=fold["train_start"],
            train_end=fold["train_end"],
            config=cluster_config,
        )
    )
fold_native_clusters = pd.concat(cluster_frames, ignore_index=True) if cluster_frames else pd.DataFrame()
fold_native_clusters.to_csv(CARD / "fold_native_teacher_rebound_clusters.csv", index=False)

target_folds = [
    {
        **fold,
        "source_window": fold["name"],
    }
    for fold in folds
]
fold_targets = build_fold_local_state_objective_training_targets(
    train_daily_paths,
    folds=target_folds,
    cluster_metrics=fold_native_clusters,
    candidate=f"{EXPERIMENT}:teacher",
)
fold_targets.to_csv(CARD / "fold_native_teacher_targets.csv", index=False)
fold_target_summary = (
    fold_targets.groupby("fold", dropna=False)
    .agg(
        rows=("datetime", "count"),
        unique_dates=("datetime", "nunique"),
        rebound_rows=("is_rebound_cluster", "sum"),
        downtrend_rows=("is_downtrend_state", "sum"),
        max_objective_weight=("objective_weight", "max"),
        mean_teacher_position=("teacher_position", "mean"),
        mean_rebound_target_position=("rebound_target_position", "mean"),
        mean_risk_target_position=("risk_target_position", "mean"),
    )
    .reset_index()
)
fold_target_summary.to_csv(CARD / "fold_target_summary.csv", index=False)
fold_target_summary

,fold,rows,unique_dates,rebound_rows,downtrend_rows,max_objective_weight,mean_teacher_position,mean_rebound_target_position,mean_risk_target_position
0,wf_2022_train,505,505,4,234,224.898212,0.208183,0.208183,0.194272
1,wf_2023_2024_train,756,756,3,478,15.719049,0.160822,0.161406,0.151255


## BC action-surface readiness

In [6]:
profiles = ["risk_budget_usage_v1", "balanced", "balanced_rule_residual_v22"]
label_rows = []
fold_label_rows = []
for profile_name in profiles:
    profile = load_profile(profile_name)
    obs, actions = build_state_objective_warm_start_dataset_from_frame(
        feature_frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    dist = pd.Series(actions).value_counts(normalize=True).sort_index()
    label_rows.append(
        {
            "profile": profile_name,
            "samples": int(len(actions)),
            "obs_dim": int(obs.shape[1]),
            "unique_actions": int(dist.shape[0]),
            "dominant_action": int(dist.idxmax()),
            "dominant_action_share": float(dist.max()),
            "action_entropy": float(-(dist * np.log(dist + 1e-12)).sum()),
        }
    )
    for fold_name, subset in fold_targets.groupby("fold"):
        fold_obs, fold_actions = build_state_objective_warm_start_dataset_from_frame(
            feature_frame,
            targets=subset,
            profile=profile,
            sample_size=0,
        )
        fold_dist = pd.Series(fold_actions).value_counts(normalize=True).sort_index()
        fold_label_rows.append(
            {
                "fold": fold_name,
                "profile": profile_name,
                "samples": int(len(fold_actions)),
                "obs_dim": int(fold_obs.shape[1]),
                "unique_actions": int(fold_dist.shape[0]),
                "dominant_action": int(fold_dist.idxmax()),
                "dominant_action_share": float(fold_dist.max()),
                "action_entropy": float(-(fold_dist * np.log(fold_dist + 1e-12)).sum()),
            }
        )

label_comparison = pd.DataFrame(label_rows).sort_values(["dominant_action_share", "action_entropy"], ascending=[True, False]).reset_index(drop=True)
fold_label_comparison = pd.DataFrame(fold_label_rows).sort_values(["fold", "dominant_action_share", "profile"]).reset_index(drop=True)
label_comparison.to_csv(CARD / "profile_label_comparison.csv", index=False)
fold_label_comparison.to_csv(CARD / "fold_profile_label_comparison.csv", index=False)
readiness_scores = score_state_objective_bc_readiness(label_comparison, target_scope="fold_local")
readiness_summary = summarize_state_objective_bc_readiness(readiness_scores)
readiness_scores.to_csv(CARD / "bc_readiness_scores.csv", index=False)
label_comparison

,profile,samples,obs_dim,unique_actions,dominant_action,dominant_action_share,action_entropy
0,risk_budget_usage_v1,755,315,5,0,0.402649,1.195994
1,balanced,755,79,9,0,0.903311,0.457479
2,balanced_rule_residual_v22,755,340,5,0,0.964238,0.197395


In [7]:
readiness_summary

{'decision': 'state_objective_bc_ready_for_training',
 'selected_profile': 'risk_budget_usage_v1',
 'selected_target_scope': 'fold_local',
 'selected_bc_readiness_gate': True,
 'selected_label_distribution_gate': True,
 'selected_fold_local_scope_gate': True,
 'selected_dominant_action_share': 0.40264900662251657,
 'selected_unique_actions': 5,
 'selected_samples': 755,
 'selected_bc_readiness_loss': 0.0,
 'next_mechanism': 'run_fold_local_bc_warm_start_then_walkforward_ppo'}

## Decision

A `training_allowed=True` result here means the next run may train a diagnostic BC warm-start / walk-forward PPO candidate from these labels. It is not a production promotion signal; promotion still requires the +30pp / <=15% MaxDD walk-forward gates after training.

In [8]:
previous_selection = json.loads((PREVIOUS_CARD / "selection.json").read_text())
current_rebound_rows = int(fold_target_summary["rebound_rows"].sum())
router_config_source_gate = bool(len(fold_router_selection) == len(folds))
label_gate = bool(readiness_summary.get("selected_bc_readiness_gate", False))
cluster_gate = current_rebound_rows > 0
training_allowed = bool(router_config_source_gate and label_gate and cluster_gate)

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "fold_native_teacher_router_targets_ready" if training_allowed else "fold_native_teacher_router_no_training",
    "promotion_allowed": False,
    "training_allowed": training_allowed,
    "selected_profile_for_next_training": readiness_summary["selected_profile"],
    "readiness_decision": readiness_summary["decision"],
    "router_config_source_gate": router_config_source_gate,
    "min_train_rebound_days": MIN_TRAIN_REBOUND_DAYS,
    "rebound_coverage_gate": bool(fold_router_selection["rebound_coverage_gate"].all()),
    "cluster_gate": cluster_gate,
    "label_distribution_gate": label_gate,
    "current_rebound_rows": current_rebound_rows,
    "previous_rebound_rows": int(previous_selection.get("current_rebound_rows", 0)),
    "teacher_source_scope": "fold_native_router_config_on_existing_base_upside_paths",
    "next_mechanism": "budget_usage_bc_warm_start_then_walkforward_ppo" if training_allowed else "repair_fold_native_teacher_source",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "fold_native_teacher_source_required": True,
        "label_distribution_gate_required": True,
    },
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://doi.org/10.1090/conm/729/14680",
    ],
    "sidecars": [
        "fold_router_top_configs.csv",
        "fold_router_selection.csv",
        "fold_train_teacher_metrics.csv",
        "fold_eval_review_metrics.csv",
        "fold_native_teacher_rebound_clusters.csv",
        "fold_native_teacher_targets.csv",
        "fold_target_summary.csv",
        "profile_label_comparison.csv",
        "fold_profile_label_comparison.csv",
        "bc_readiness_scores.csv",
        "summary.json",
    ],
}
summary = {
    "selection": selection,
    "fold_router_selection": fold_router_selection.to_dict(orient="records"),
    "train_metrics": train_metrics.to_dict(orient="records"),
    "eval_review_metrics": eval_review_metrics.to_dict(orient="records"),
    "fold_target_summary": fold_target_summary.to_dict(orient="records"),
    "label_comparison": label_comparison.to_dict(orient="records"),
    "readiness_summary": readiness_summary,
    "conclusion": [
        "Router config selection is now performed separately inside each train fold; eval dates are review-only.",
        f"Fold-native teacher targets contain {current_rebound_rows} rebound rows.",
        f"{readiness_summary['selected_profile']} remains the selected BC action surface.",
        "The next valid step is diagnostic budget-usage BC warm-start followed by walk-forward PPO and full promotion gates." if training_allowed else "Training remains blocked by fold-native source or label readiness gates.",
    ],
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True))
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True, default=str))
selection

{'experiment': 'state_objective_fold_native_teacher_router_v1',
 'created_at': '2026-05-25T11:30:35.105652+00:00',
 'decision': 'fold_native_teacher_router_targets_ready',
 'promotion_allowed': False,
 'training_allowed': True,
 'selected_profile_for_next_training': 'risk_budget_usage_v1',
 'readiness_decision': 'state_objective_bc_ready_for_training',
 'router_config_source_gate': True,
 'min_train_rebound_days': 3,
 'rebound_coverage_gate': True,
 'cluster_gate': True,
 'label_distribution_gate': True,
 'current_rebound_rows': 7,
 'previous_rebound_rows': 25,
 'teacher_source_scope': 'fold_native_router_config_on_existing_base_upside_paths',
 'next_mechanism': 'budget_usage_bc_warm_start_then_walkforward_ppo',
 'acceptance_gate_for_next_run': {'relative_return_min_pp': 30.0,
  'max_dd_max_pct': 15.0,
  'walkforward_required': True,
  'fold_native_teacher_source_required': True,
  'label_distribution_gate_required': True},
 'external_sources': ['https://github.com/microsoft/qlib',
  '